In [37]:
import pandas as pd
import numpy as np
import geopandas as gpd

In [38]:
df = pd.read_csv(r'C:\Users\HP\Downloads\Crime_Data_from_2020_to_Present_20260220.csv')

In [39]:
print(df.shape)
print(df.head())

(1004991, 28)
       DR_NO                Date Rptd                 DATE OCC  TIME OCC  \
0  211507896  2021 Apr 11 12:00:00 AM  2020 Nov 07 12:00:00 AM       845   
1  201516622  2020 Oct 21 12:00:00 AM  2020 Oct 18 12:00:00 AM      1845   
2  240913563  2024 Dec 10 12:00:00 AM  2020 Oct 30 12:00:00 AM      1240   
3  210704711  2020 Dec 24 12:00:00 AM  2020 Dec 24 12:00:00 AM      1310   
4  201418201  2020 Oct 03 12:00:00 AM  2020 Sep 29 12:00:00 AM      1830   

   AREA    AREA NAME  Rpt Dist No  Part 1-2  Crm Cd  \
0    15  N Hollywood         1502         2     354   
1    15  N Hollywood         1521         1     230   
2     9     Van Nuys          933         2     354   
3     7     Wilshire          782         1     331   
4    14      Pacific         1454         1     420   

                                         Crm Cd Desc  ... Status  Status Desc  \
0                                  THEFT OF IDENTITY  ...     IC  Invest Cont   
1     ASSAULT WITH DEADLY WEAPON, AG

In [40]:
cols_keep = [
    "DATE OCC",
    "TIME OCC",
    "Crm Cd",
    "Crm Cd Desc",
    "Part 1-2",
    "AREA NAME",
    "LAT",
    "LON"
]

df = df[cols_keep].copy()

In [41]:
df["is_serious_crime"] = (df["Part 1-2"] == 1).astype(int)

In [42]:
df["Part 1-2"].value_counts()

Part 1-2
1    602645
2    402346
Name: count, dtype: int64

In [ ]:
df["Crm Cd Desc"] = df["Crm Cd Desc"].str.lower().str.strip()

In [21]:
print(edges[["local_severity_norm"]].head())

   local_severity_norm
0             0.669799
1             0.664286
2             0.671605
3             0.663158
4             0.667692


In [43]:
df["DATE OCC"] = pd.to_datetime(df["DATE OCC"], errors="coerce")

df = df[
    (df["DATE OCC"].dt.year >= 2023) &
    (df["DATE OCC"].dt.year <= 2025)
].copy()

C:\Users\HP\AppData\Local\Temp\ipykernel_67696\2299812467.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["DATE OCC"] = pd.to_datetime(df["DATE OCC"], errors="coerce")


In [44]:
print(df.shape)

(360009, 9)


In [ ]:
df["TIME OCC"] = df["TIME OCC"].astype(int).astype(str).str.zfill(4)
df["TIME OCC"] = pd.to_datetime(df["TIME OCC"], format="%H%M").dt.time
print(df["TIME OCC"].head(5))

644982    11:40:00
644983    16:30:00
644984    10:30:00
644985    08:16:00
644986    01:10:00
Name: TIME OCC, dtype: object


In [46]:
df["DATETIME"] = pd.to_datetime(
    df["DATE OCC"].astype(str) + " " + df["TIME OCC"].astype(str)
)
print(df[["DATE OCC","TIME OCC","DATETIME"]].head())

         DATE OCC  TIME OCC            DATETIME
644982 2023-06-17  11:40:00 2023-06-17 11:40:00
644983 2023-02-11  16:30:00 2023-02-11 16:30:00
644984 2023-09-19  10:30:00 2023-09-19 10:30:00
644985 2023-04-10  08:16:00 2023-04-10 08:16:00
644986 2023-12-01  01:10:00 2023-12-01 01:10:00


In [47]:
df["HOUR"] = df["DATETIME"].dt.hour
df["DAY_OF_WEEK"] = df["DATETIME"].dt.dayofweek
df["MONTH"] = df["DATETIME"].dt.month

df["IS_WEEKEND"] = df["DAY_OF_WEEK"].isin([5,6]).astype(int)

In [48]:
df["hour_sin"] = np.sin(2*np.pi*df["HOUR"]/24)
df["hour_cos"] = np.cos(2*np.pi*df["HOUR"]/24)

In [49]:
df["Crm Cd Desc"] = df["Crm Cd Desc"].str.lower().str.strip()
df["AREA NAME"] = df["AREA NAME"].str.lower().str.strip()

In [50]:
df["crime_category"] = "disorder"

df.loc[df["Crm Cd Desc"].str.contains(
    "rape|sex|sodomy|lewd|indecent", na=False
), "crime_category"] = "sexual"

df.loc[df["Crm Cd Desc"].str.contains(
    "homicide|assault|robbery|battery|kidnapping|criminal threats", na=False
), "crime_category"] = "violent"

df.loc[df["Crm Cd Desc"].str.contains(
    "burglary|theft|stolen|larceny|arson|vandalism|shoplifting|identity", na=False
), "crime_category"] = "property"

In [51]:
df = df[(df["LAT"] != 0) & (df["LON"] != 0)]

In [ ]:
gdf = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df["LON"], df["LAT"]),
    crs="EPSG:4326"
)

In [53]:
gdf = gdf.to_crs("EPSG:32611")

gdf["x"] = gdf.geometry.x
gdf["y"] = gdf.geometry.y

In [54]:
severity_map = {
    "violent": 5,
    "sexual": 5,
    "property": 3,
    "disorder": 1
}

gdf["severity"] = gdf["crime_category"].map(severity_map)

In [55]:
gdf.to_pickle("crime_preprocessed.pkl")